### Imports

In [1]:
import numpy as np
import pyproj as proj

from whale_gunshot_localization.utils.experimental import MultilaterationOpt, Localizer, ParLocalizer
import whale_gunshot_localization.sim_tools.sim_datagen as sim_datagen
from whale_gunshot_localization import config, PROJECT_ROOT_DIR

### Generate Data

In [2]:
# derive TOSSIT locations relative to the first from the lat/lons
TOSSIT_latlons = np.asarray([config['TOSSIT']['TOSSIT_lat'], config['TOSSIT']['TOSSIT_lon']]).T
pargs = proj.Proj(proj="aeqd", lat_0=TOSSIT_latlons[0, 0], lon_0=TOSSIT_latlons[0, 1], datum="WGS84", units="m")
xs, ys = pargs(TOSSIT_latlons[:,1], TOSSIT_latlons[:,0])
TOSSIT_locations = np.asarray([-ys, xs]).T

data_gen_params = dict(num_delete=0, 
                       rng=np.random.default_rng(1524), 
                       in_sensors=True,
                       TOSSIT_locations=TOSSIT_locations)

measurements, source_associations, TOSSIT_associations, source_locs = sim_datagen.generate_measurements(num_sources=3, 
                                                                                                        var=0, 
                                                                                                        **data_gen_params)

In [5]:
Ls = Localizer(k=5, multilat=MultilaterationOpt(method_thresh=float('inf')), consistency_thresh=500, prune=False, TOSSIT_locations=TOSSIT_locations, min_assoc_size=8)

In [6]:
%%time
success = Ls.set_measurements(measurements, adaptive=False)

CPU times: user 1min 4s, sys: 174 ms, total: 1min 4s
Wall time: 1min 4s


In [5]:
Lp = ParLocalizer(k=5, multilat=MultilaterationOpt(method_thresh=float('inf')), consistency_thresh=500, TOSSIT_locations=TOSSIT_locations, min_assoc_size=8)

In [9]:
%%time
success = Lp.set_measurements(measurements, adaptive=False)

CPU times: user 603 ms, sys: 1.81 s, total: 2.41 s
Wall time: 16.2 s
